# DB Scout Language ShiftResearch question: **How has scout language for top-tier defensive backs shifted from raw athleticism to scheme/coverage detail between 2014 and 2025?**The notebook: (a) extracts DB text per year, (b) vectorizes with 1-3 gram TF-IDF, (c) outputs per-year term tables, (d) aggregates themes, (e) plots the transitions, and (f) documents methodology (stop words, lemmatization, cohort selection).

In [ ]:

import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

try:
    import nltk
    from nltk.stem import WordNetLemmatizer
    nltk.download('wordnet', quiet=True)
    lemmatizer = WordNetLemmatizer()
except Exception:
    lemmatizer = None


In [ ]:

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent

text_file = root / 'data' / 'processed' / 'draft_enriched_with_contracts.csv'
df = pd.read_csv(text_file)
text_cols = ['overview', 'strengths', 'weaknesses']
df[text_cols] = df[text_cols].fillna('')
df['all_text'] = df[text_cols].agg(' '.join, axis=1)

df = df[df['Pos_Group'] == 'DB'].copy()
df = df[df['year'].between(2014, 2025)]
df = df[df['grade'].notna()].copy()
df['year'] = df['year'].astype(int)

custom_stop = {
    'ball', 'play', 'plays', 'field', 'team', 'teams', 'game', 'games',
    'player', 'players', 'coverage', 'good'
}


def clean_and_lemmatize(text: str) -> str:
    text = text.lower()
    tokens = re.findall(r"[a-z']+", text)
    tokens = [tok for tok in tokens if len(tok) > 1 and tok not in custom_stop]
    if lemmatizer:
        tokens = [lemmatizer.lemmatize(tok) for tok in tokens]
    return ' '.join(tokens)

df['processed_text'] = df['all_text'].apply(clean_and_lemmatize)

cohort_pct = 0.25
min_players = 5
best_docs = []
for year, group in df.groupby('year'):
    if len(group) < min_players:
        continue
    cutoff = group['grade'].quantile(1 - cohort_pct, interpolation='higher')
    top = group[group['grade'] >= cutoff]
    if len(top) < min_players:
        top = group.nlargest(min_players, 'grade')
    doc_text = ' '.join(top['processed_text'])
    if not doc_text.strip():
        continue
    best_docs.append({
        'year': year,
        'doc': doc_text,
        'n_players': len(top)
    })

best_df = pd.DataFrame(best_docs).sort_values('year')
best_df


In [ ]:

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    max_features=5000,
    min_df=2,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)
dtm = vectorizer.fit_transform(best_df['doc'])
features = vectorizer.get_feature_names_out()
rows = []
for idx, row in best_df.iterrows():
    weights = dtm[idx].toarray().ravel()
    top_idx = np.argsort(weights)[-30:][::-1]
    for term_idx in top_idx:
        weight = weights[term_idx]
        if weight <= 0:
            continue
        rows.append({
            'year': row['year'],
            'term': features[term_idx],
            'weight': float(weight),
            'n_players': row['n_players']
        })
tf_idf_df = pd.DataFrame(rows).sort_values(['year', 'weight'], ascending=[True, False])
print('TF-IDF entries:', len(tf_idf_df))
tf_idf_df.head()


In [ ]:

top_by_year = tf_idf_df.groupby('year').head(10).reset_index(drop=True)
top_by_year


In [ ]:

bucket_terms = {
    'Athleticism': ['speed', 'athletic', 'quick', 'burst', 'agility', 'explosive'],
    'Coverage': ['coverage', 'zone', 'man coverage', 'press', 'coverage responsibilities', 'match release'],
    'Physical Tools': ['size', 'length', 'hands', 'strength', 'catch point', 'run support', 'tackle'],
    'Technique': ['ball skills', 'ball production', 'route breaks', 'route anticipation', 'press man', 'play recognition', 'open field']
}

bucket_df = tf_idf_df.copy()

def assign_bucket(term: str) -> str:
    lower = term.lower()
    for bucket, keywords in bucket_terms.items():
        if any(key in lower for key in keywords):
            return bucket
    return 'Other'

bucket_df['bucket'] = bucket_df['term'].apply(assign_bucket)
summary = (
    bucket_df.groupby(['year', 'bucket'])['weight']
    .sum()
    .reset_index()
)

totals = summary.groupby('year')['weight'].sum().rename('year_total')
summary = summary.merge(totals, on='year')
summary['fraction'] = summary['weight'] / summary['year_total']
summary


In [ ]:

sns.set_style('whitegrid')
plt.figure(figsize=(10, 4))
sns.lineplot(data=summary, x='year', y='weight', hue='bucket', marker='o')
plt.title('TF-IDF weight per thematic bucket')
plt.ylabel('Weight')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.lineplot(data=summary, x='year', y='fraction', hue='bucket', marker='o')
plt.title('Fractional share per bucket')
plt.ylabel('Share')
plt.tight_layout()
plt.show()


In [ ]:

period_docs = {
    'early': ' '.join(best_df[best_df['year'] <= 2018]['doc']),
    'late': ' '.join(best_df[best_df['year'] > 2018]['doc'])
}
count_vectorizer = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)
count_matrix = count_vectorizer.fit_transform(period_docs.values())
terms = count_vectorizer.get_feature_names_out()
counts = pd.DataFrame(count_matrix.toarray().T, index=terms, columns=list(period_docs.keys()))
counts = counts[counts.sum(axis=1) > 5]
counts['llr'] = 0.0
counts['freq_ratio'] = 1.0
for term in counts.index:
    k11 = counts.at[term, 'early']
    k21 = counts.at[term, 'late']
    k12 = counts['early'].sum() - k11
    k22 = counts['late'].sum() - k21
    total = k11 + k12 + k21 + k22
    row_sums = [k11 + k12, k21 + k22]
    col_sums = [k11 + k21, k12 + k22]
    expected = [r * c / total for r in row_sums for c in col_sums]
    observed = [k11, k12, k21, k22]
    ll = 0
    for obs, exp in zip(observed, expected):
        if obs > 0 and exp > 0:
            ll += obs * math.log(obs / exp)
    counts.at[term, 'llr'] = 2 * ll
    denom = (k11 / max(counts['early'].sum(), 1))
    if denom > 0:
        counts.at[term, 'freq_ratio'] = (k21 / max(counts['late'].sum(), 1)) / denom
ll_df = counts.sort_values('llr', ascending=False).head(15)
ll_df[['llr', 'freq_ratio']]


In [ ]:

early_quote = df[df['year'] == 2015]['processed_text'].head(1)
late_quote = df[df['year'] == 2024]['processed_text'].head(1)
print('2015 snippet:', early_quote.iloc[0] if not early_quote.empty else 'n/a')
print('
2024 snippet:', late_quote.iloc[0] if not late_quote.empty else 'n/a')


## Methodology & Findings- Preprocessing: combined `overview`, `strengths`, and `weaknesses`, removed custom stop words, lowercased/lemmatized when possible, and kept the top 25% DBs per year (min 5 players).- TF-IDF: 1-3 gram vectorization (min_df=2, English stop words) produces per-year term tables.- Themes: Athleticism, Coverage, Physical Tools, and Technique buckets track term weights and share over time.- Validation: log-likelihood/frequency ratios highlight the most shifted n-grams between 2014-2018 and 2019-2025; sample text snippets anchor the narrative.
Next steps: save the TF-IDF tables out to CSV, stratify cornerbacks vs. safeties, and add annotated quotes to the plots.
